# Trading Pipeline — Exploration
Inspect candles, technicals, prompts, and DB. Import-only from the flat scripts.

In [ ]:
import os
os.environ['DATA_SOURCE'] = 'csv'   # change to 'yfinance' if needed

import json, sqlite3, pandas as pd
from technicals import TechnicalsAnalyzer
from pipeline   import build_prompt, DB_PATH

SYMBOL = 'CANBK'
DATE   = '2024-10-10'

ta = TechnicalsAnalyzer()
print('Ready')

## 1 — Candles

In [ ]:
df = ta.load_candles(SYMBOL, DATE, lookback_days=30)
df.tail(10)

In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df.index, df['close'], label='Close')
ax1.plot(df.index, df['close'].ewm(span=50,  adjust=False).mean(), '--', lw=1, label='EMA50')
ax1.plot(df.index, df['close'].ewm(span=200, adjust=False).mean(), '--', lw=1, label='EMA200')
ax1.set_title(f'{SYMBOL}  {DATE}  (30-day lookback)')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.bar(df.index, df['volume'] / 1e6, color='steelblue', alpha=0.6)
ax2.set_ylabel('Vol (M)'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2 — Technicals

In [ ]:
tech = ta.compute(df)
vc   = ta.vix_context(DATE)
zc   = ta.compute_zones(SYMBOL, DATE)

for k, v in tech.items():
    if k not in ('recent_candles',):
        print(f'  {k:25s}: {v}')

In [ ]:
print('VIX:', vc)
print('Zones:', json.dumps(zc, indent=2))

## 3 — LLM Prompt Preview

In [ ]:
prompt = build_prompt(SYMBOL, DATE, tech, vc, zc)
print(prompt)
print(f'\n~{len(prompt)//4} tokens')

## 4 — DB Contents

In [ ]:
with sqlite3.connect(DB_PATH) as con:
    df_db = pd.read_sql(
        'SELECT symbol, decision_date, action, entry_price, stop_loss, take_profit, '
        'exit_reason, pnl, confidence FROM trade_log ORDER BY decision_date DESC LIMIT 20',
        con)
df_db

In [ ]:
with sqlite3.connect(DB_PATH) as con:
    stats = pd.read_sql(
        "SELECT action, exit_reason, COUNT(*) n, ROUND(AVG(pnl),2) avg_pnl, "
        "ROUND(SUM(pnl),2) total_pnl FROM trade_log GROUP BY action, exit_reason",
        con)
stats

## 5 — Training Example Preview

In [ ]:
from splitter import load_trades, build_example

trades = load_trades(DB_PATH)
print(f'{len(trades)} trades in DB')

if trades:
    ex = build_example(trades[0])
    for msg in ex['messages']:
        print(f"\n--- {msg['role'].upper()} ---")
        print(msg['content'][:600])

## 6 — Forward Candles (outcome check)

In [ ]:
fwd = ta.load_forward(SYMBOL, DATE, n=5)
fwd[['open','high','low','close','volume']]

## 7 — Split DB → JSONL (run before training)

In [ ]:
from splitter import build_dataset

counts = build_dataset(
    db_path      = DB_PATH,
    out_path     = "data/trading.jsonl",
    min_confidence = 0.5,
    min_rr         = 1.5,
    val_ratio      = 0.1,
    test_ratio     = 0.1,
)
print(counts)

## 8 — Train (PyTorch LoRA — MPS / CUDA / CPU)
> Runs overnight (~8-12h on M5 for 3 epochs). Output streamed to `output/train.log`. Set `EPOCHS=1` for a quick smoke-test.

In [ ]:
import subprocess, pathlib, os

TRAIN_JSONL = "data/trading_train.jsonl"
VAL_JSONL   = "data/trading_val.jsonl"
MODEL       = os.environ.get("BASE_MODEL", "nvidia/Nemotron-Mini-4B-Instruct")
OUTPUT_DIR  = "output/nemotron-lora"
LOG_FILE    = "output/train.log"
EPOCHS      = 1    # set to 3 for full training; 1 for smoke-test

pathlib.Path("output").mkdir(exist_ok=True)

cmd = [
    "python", "train.py",
    "--train",      TRAIN_JSONL,
    "--val",        VAL_JSONL,
    "--model",      MODEL,
    "--output_dir", OUTPUT_DIR,
]
env = os.environ.copy()
env["EPOCHS"] = str(EPOCHS)

print("Starting training — logging to", LOG_FILE)
print("Command:", " ".join(cmd))
print(f"Device: MPS (Apple Silicon) | Epochs: {EPOCHS}")
print("View live metrics: run 'mlflow ui' in terminal → http://localhost:5000")

with open(LOG_FILE, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

print(f"\nPID {proc.pid} — re-run the loss graph cell below to track progress")
print("After training finishes, merge the adapter:")
print(f"  python train.py --merge --adapter {OUTPUT_DIR} --base_model {MODEL}")


## 9 — Loss Graph (re-run any time to refresh)

In [ ]:
import re, matplotlib.pyplot as plt, json

# HuggingFace Trainer logs to stdout — each step looks like:
#   {"loss": 2.345, "grad_norm": ..., "learning_rate": ..., "epoch": 0.5}
# or formatted as:
#   {'loss': 2.345, 'grad_norm': ..., 'epoch': 0.5}
train_steps, train_loss = [], []
eval_steps,  eval_loss  = [], []
step = 0

try:
    with open(LOG_FILE) as f:
        for line in f:
            line = line.strip()
            # HF Trainer JSON log lines
            for pat in (r'\{.*"loss".*\}', r"\{.*'loss'.*\}"):
                m = re.search(pat, line)
                if m:
                    try:
                        d = json.loads(m.group().replace("'", '"'))
                        step += 1
                        if "loss" in d:
                            train_steps.append(step)
                            train_loss.append(float(d["loss"]))
                        if "eval_loss" in d:
                            eval_steps.append(step)
                            eval_loss.append(float(d["eval_loss"]))
                    except Exception:
                        pass
            # Plain text fallback: "{'loss': X, ...}"
            m = re.search(r"'loss':\s*([\d.]+)", line)
            if m and not train_loss:
                step += 1
                train_steps.append(step)
                train_loss.append(float(m.group(1)))
            m = re.search(r"'eval_loss':\s*([\d.]+)", line)
            if m:
                eval_steps.append(step)
                eval_loss.append(float(m.group(1)))
except FileNotFoundError:
    print(f"Log not found: {LOG_FILE} — run the training cell first")

if train_loss:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(train_steps, train_loss, label="Train loss", color="steelblue")
    if eval_loss:
        ax.plot(eval_steps, eval_loss, label="Val loss", color="tomato", marker="o", ms=4)
    ax.set_xlabel("Step"); ax.set_ylabel("Loss")
    ax.set_title("Training Loss — HuggingFace Trainer")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"Last train loss: {train_loss[-1]:.4f}  |  steps so far: {train_steps[-1]}")
    if eval_loss:
        print(f"Last val   loss: {eval_loss[-1]:.4f}")
else:
    print("No loss data yet — training may still be starting up or logging hasn't begun")


## 10 — Evaluate (after training finishes)

In [ ]:
import json
from pathlib import Path
from evaluate import evaluate_model_quality, trading_metrics, pnls_from_test_meta

TEST_JSONL   = "data/trading_test.jsonl"
FUSED_MODEL  = None   # set to "output/nemotron-lora/fused_model" after training + merge

# ── Self-check (no model required) ──────────────────────────────────────────
# Echoes stored answers → expect ~100% on all quality metrics. Good harness test.
rows = [json.loads(l) for l in open(TEST_JSONL) if l.strip()]
_answers = {}
for _ex in rows:
    _u = next((m["content"] for m in _ex["messages"] if m["role"] == "user"), "")
    _a = next((m["content"] for m in _ex["messages"] if m["role"] == "assistant"), "")
    _answers[_u] = _a

def _stored_gen(system, user):
    return _answers.get(user, "")

gen_fn = _stored_gen
label  = "self-check (stored answers)"

# ── Use fine-tuned HuggingFace model ────────────────────────────────────────
if FUSED_MODEL and Path(FUSED_MODEL).exists():
    from evaluate import hf_generate_factory
    gen_fn = hf_generate_factory(FUSED_MODEL)
    label  = f"HF fused model: {FUSED_MODEL}"

print(f"Evaluating: {label}")

# ── A) Model quality ──────────────────────────────────────────────────────────
q = evaluate_model_quality(TEST_JSONL, gen_fn)
print("\nMODEL QUALITY (held-out test):")
for k, v in q.items():
    print(f"  {k:20s}: {v}")

# ── B) Trading metrics from recorded outcomes ────────────────────────────────
pnls = pnls_from_test_meta(TEST_JSONL)
if pnls:
    tm = trading_metrics(pnls)
    print("\nTRADING METRICS (recorded P&L on test dates):")
    for k, v in tm.items():
        flag = " ✓" if k == "significant_95" and v else ""
        print(f"  {k:22s}: {v}{flag}")
else:
    print("\nNo P&L data in test meta — run splitter.py first, then re-check")
